In [ ]:
import os
import librosa
from pathlib import Path
import json
import pandas as pd
from wer_evaluation import _parse_transcript_literal

# Top 10 clips where MRS_WER is lower than WER, with clip id and both transcripts.
if 'df_comparison' not in globals():
    raise NameError('df_comparison is not loaded. Run the earlier comparison cell first.')

required_cols = {'clip_id', 'WER', 'MRS_WER'}
missing = required_cols - set(df_comparison.columns)
if missing:
    raise ValueError(f'df_comparison is missing columns: {sorted(missing)}')

def flatten_segments(value):
    segments = _parse_transcript_literal(value) if not isinstance(value, list) else value
    parts = []
    for seg in segments:
        if isinstance(seg, (list, tuple)) and len(seg) >= 2:
            parts.append(str(seg[1]).strip())
        elif isinstance(seg, str):
            parts.append(seg.strip())
    return ' '.join(part for part in parts if part).strip()

# Load reference transcripts from the manifest.
manifest_path_candidates = [
    Path('Output/manifest.csv'),
    Path('../Output/manifest.csv'),
    Path('/home/jamin/Year3Proj/Output/manifest.csv'),
]
manifest_path = next((p for p in manifest_path_candidates if p.exists()), None)
if manifest_path is None:
    raise FileNotFoundError('Could not find Output/manifest.csv')

manifest_df = pd.read_csv(manifest_path)
if 'clip_id' not in manifest_df.columns or 'transcript' not in manifest_df.columns:
    raise ValueError('manifest.csv must contain clip_id and transcript columns')

manifest_lookup = {row['clip_id']: flatten_segments(row['transcript']) for _, row in manifest_df.iterrows()}

# Load wav2vec2 hypotheses from ASR_transcriptions.json and flatten them into text.
asr_path_candidates = [
    Path('ASR_transcriptions.json'),
    Path('../ASR_transcriptions.json'),
    Path('/home/jamin/Year3Proj/ASR_transcriptions.json'),
]
asr_path = next((p for p in asr_path_candidates if p.exists()), None)
if asr_path is None:
    raise FileNotFoundError('Could not find ASR_transcriptions.json')

with open(asr_path, 'r', encoding='utf-8') as f:
    asr_payload = json.load(f)

wav_lookup = {}
for clip, item in asr_payload.items():
    transcript_block = item.get('transcript', {}) if isinstance(item, dict) else {}
    wav_lookup[clip] = flatten_segments(transcript_block.get('wav2vec2'))

# Build the top-10 table.
summary_df = df_comparison.copy()
summary_df['Difference'] = summary_df['WER'] - summary_df['MRS_WER']
summary_df = summary_df[summary_df['Difference'] > 0].sort_values('Difference', ascending=False).head(10).copy()
summary_df['Reference Transcript'] = summary_df['clip_id'].map(manifest_lookup)
summary_df['Wav2Vec2 Transcript'] = summary_df['clip_id'].map(wav_lookup)

# Keep the output compact and easy to scan.
out = summary_df[[
    'clip_id',
    'WER',
    'MRS_WER',
    'Difference',
    'Reference Transcript',
    'Wav2Vec2 Transcript',
]

print(f'Loaded manifest from: {manifest_path.resolve()}')
print(f'Loaded ASR transcripts from: {asr_path.resolve()}')
print('\nTop 10 clips where MRS_WER < WER:')
display(out.reset_index(drop=True))
    
    for result in data['results']:
        # Check if successful
        if result.get('status') != 'success':
            continue
        
        clip_id = result['clip_id']
        overlap = extract_overlap_from_clip_id(clip_id)
        
        # Only interested in 0 overlap clips
        if overlap is None or overlap != 0.0:
            continue
        
        metrics = result.get('metrics', {})
        wer = metrics.get('wer')
        mrs_wer = metrics.get('mrs_wer')
        
        # Check if wer and mrs_wer are different
        if wer is not None and mrs_wer is not None:
            if abs(wer - mrs_wer) > 1e-10:  # floating point comparison
                zero_overlap_diffs[model_name].append({
                    'clip_id': clip_id,
                    'wer': wer,
                    'mrs_wer': mrs_wer,
                    'difference': abs(wer - mrs_wer),
                    'ref_segments': result.get('ref_segments'),
                    'hyp_segments': result.get('hyp_segments'),
                    'is_segmented': result.get('is_segmented'),
                })

print("=" * 100)
print("CLIPS WITH 0 OVERLAP THAT HAVE DIFFERENT WER AND MRS_WER VALUES")
print("=" * 100)

total_clips = 0

for model_name in sorted(zero_overlap_diffs.keys()):
    clips = zero_overlap_diffs[model_name]
    print(f"\n{model_name.upper()} - {len(clips)} clips found")
    print("-" * 100)
    print(f"{'Clip ID':<40} {'WER':<12} {'MRS_WER':<12} {'Difference':<12} {'Ref Seg':<10} {'Hyp Seg':<10}")
    print("-" * 100)
    
    for clip in sorted(clips, key=lambda x: x['difference'], reverse=True):
        print(f"{clip['clip_id']:<40} {clip['wer']:<12.6f} {clip['mrs_wer']:<12.6f} {clip['difference']:<12.6f} {clip['ref_segments']:<10} {clip['hyp_segments']:<10}")
        total_clips += 1

print("\n" + "=" * 100)
print(f"TOTAL CLIPS WITH 0 OVERLAP AND DIFFERENT WER/MRS_WER: {total_clips}")
print("=" * 100)

# Save to JSON file
output_file = '/home/jamin/Year3Proj/Output/zero_overlap_diff_wer.json'
with open(output_file, 'w') as f:
    json.dump(zero_overlap_diffs, f, indent=2)
print(f"\nResults saved to: {output_file}")

In [4]:
import matplotlib.pyplot as plt

print("\n" + "=" * 120)
print("DETAILED ANALYSIS: Why WER vs MRS_WER Differ")
print("=" * 120)

# Analyze clips where WER and MRS_WER differ the most
print("\nClips with LARGEST differences (WER > MRS_WER suggests speaker structure helps):")
df_wer_higher = df_comparison[df_comparison['WER'] > df_comparison['MRS_WER']].copy()
df_wer_higher['Abs_Diff'] = df_wer_higher['Difference']
top_wer_higher = df_wer_higher.nlargest(5, 'Abs_Diff')
print(top_wer_higher[['Model', 'Clip ID', 'WER', 'MRS_WER', 'Difference']].to_string(index=False))

print("\n\nClips where MRS_WER > WER (unexpected - suggests speaker structure hurts):")
df_mrs_higher = df_comparison[df_comparison['MRS_WER'] > df_comparison['WER']].copy()
df_mrs_higher['Abs_Diff'] = df_mrs_higher['MRS_WER'] - df_mrs_higher['WER']
top_mrs_higher = df_mrs_higher.nlargest(5, 'Abs_Diff')
print(top_mrs_higher[['Model', 'Clip ID', 'WER', 'MRS_WER']].to_string(index=False))

# Statistics
print("\n" + "-" * 120)
print("Summary Statistics:")
print("-" * 120)
wer_higher_count = (df_comparison['WER'] > df_comparison['MRS_WER']).sum()
mrs_higher_count = (df_comparison['MRS_WER'] > df_comparison['WER']).sum()
print(f"Clips where WER > MRS_WER: {wer_higher_count} ({wer_higher_count/len(df_comparison)*100:.1f}%)")
print(f"Clips where MRS_WER > WER: {mrs_higher_count} ({mrs_higher_count/len(df_comparison)*100:.1f}%)")

print(f"\nAverage WER: {df_comparison['WER'].mean():.6f}")
print(f"Average MRS_WER: {df_comparison['MRS_WER'].mean():.6f}")
print(f"Average difference (|WER - MRS_WER|): {df_comparison['Difference'].mean():.6f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: WER vs MRS_WER
axes[0].scatter(df_comparison['WER'], df_comparison['MRS_WER'], alpha=0.6)
axes[0].plot([0, 1], [0, 1], 'r--', label='WER = MRS_WER')
axes[0].set_xlabel('WER', fontsize=12)
axes[0].set_ylabel('MRS_WER', fontsize=12)
axes[0].set_title('WER vs MRS_WER Comparison\n(Points above diagonal = MRS_WER worse)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution comparison
axes[1].hist([df_comparison['WER'], df_comparison['MRS_WER']], label=['WER', 'MRS_WER'], bins=20, alpha=0.7)
axes[1].set_xlabel('Error Rate', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of WER vs MRS_WER', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/home/jamin/Year3Proj/Output/plot/wer_vs_mrs_wer_analysis.png', dpi=150, bbox_inches='tight')
print("\n" + "=" * 120)
print("Plot saved to: /home/jamin/Year3Proj/Output/plot/wer_vs_mrs_wer_analysis.png")
print("=" * 120)


DETAILED ANALYSIS: Why WER vs MRS_WER Differ

Clips with LARGEST differences (WER > MRS_WER suggests speaker structure helps):


KeyError: 'Difference'

In [38]:
# Wav2vec2: compare WER vs MRS_WER separately for no-overlap and overlap clips (all successful rows).
import json
from pathlib import Path
import numpy as np
import pandas as pd

def extract_overlap_from_clip_id(clip_id: str):
    parts = str(clip_id).split("_")
    if len(parts) >= 3:
        try:
            return float(parts[2])
        except (ValueError, TypeError):
            return np.nan
    return np.nan

wav2vec2_path = Path('/home/jamin/Year3Proj/WER_Results_by_Model/wav2vec2_wer_results.json')
with open(wav2vec2_path, 'r', encoding='utf-8') as f:
    wav_payload = json.load(f)

rows = []
for item in wav_payload.get('results', []):
    if item.get('status') != 'success':
        continue
    metrics = item.get('metrics', {})
    wer = metrics.get('wer', np.nan)
    mrs_wer = metrics.get('mrs_wer', np.nan)
    if pd.isna(wer) or pd.isna(mrs_wer):
        continue

    clip_id = item.get('clip_id')
    overlap = extract_overlap_from_clip_id(clip_id)
    overlap_group = 'no_overlap' if np.isclose(overlap, 0.0, equal_nan=False) else 'overlap'

    rows.append({
        'clip_id': clip_id,
        'overlap': overlap,
        'overlap_group': overlap_group,
        'wer': float(wer),
        'mrs_wer': float(mrs_wer),
        'wer_minus_mrs_wer': float(wer) - float(mrs_wer),
    })

wav_df = pd.DataFrame(rows)
if wav_df.empty:
    print('No successful wav2vec2 rows with both WER and MRS_WER were found.')
else:
    wav_df['abs_diff'] = wav_df['wer_minus_mrs_wer'].abs()

    summary = (
        wav_df.groupby('overlap_group', as_index=False)
        .agg(
            clips=('clip_id', 'count'),
            wer_mean=('wer', 'mean'),
            mrs_wer_mean=('mrs_wer', 'mean'),
            diff_mean=('wer_minus_mrs_wer', 'mean'),
            diff_median=('wer_minus_mrs_wer', 'median'),
            abs_diff_mean=('abs_diff', 'mean'),
            pct_wer_gt_mrs=('wer_minus_mrs_wer', lambda s: 100.0 * (s > 0).mean()),
            pct_wer_lt_mrs=('wer_minus_mrs_wer', lambda s: 100.0 * (s < 0).mean()),
            pct_wer_eq_mrs=('wer_minus_mrs_wer', lambda s: 100.0 * np.isclose(s, 0.0).mean()),
        )
        .sort_values('overlap_group')
    )

    print('WAV2VEC2: WER vs MRS_WER split by overlap group')
    display(summary.round(4))

    print('\nTop 10 largest |WER - MRS_WER| clips for no_overlap:')
    display(
        wav_df[wav_df['overlap_group'] == 'no_overlap']
        .nlargest(10, 'abs_diff')[['clip_id', 'overlap', 'wer', 'mrs_wer', 'wer_minus_mrs_wer', 'abs_diff']]
        .reset_index(drop=True)
    )

    print('\nTop 10 largest |WER - MRS_WER| clips for overlap:')
    display(
        wav_df[wav_df['overlap_group'] == 'overlap']
        .nlargest(10, 'abs_diff')[['clip_id', 'overlap', 'wer', 'mrs_wer', 'wer_minus_mrs_wer', 'abs_diff']]
        .reset_index(drop=True)
    )

WAV2VEC2: WER vs MRS_WER split by overlap group


,overlap_group,clips,wer_mean,mrs_wer_mean,diff_mean,diff_median,abs_diff_mean,pct_wer_gt_mrs,pct_wer_lt_mrs,pct_wer_eq_mrs
0,no_overlap,120,0.2358,0.2403,-0.0045,0.0000,0.0074,12.5000,18.3333,69.1667
1,overlap,3600,0.4323,0.4849,-0.0526,-0.0415,0.0581,15.3611,78.2500,6.3889



Top 10 largest |WER - MRS_WER| clips for no_overlap:


,clip_id,overlap,wer,mrs_wer,wer_minus_mrs_wer,abs_diff
0,mix_0000079_0.00_2_0_P,0.0,0.700637,0.783439,-0.082803,0.082803
1,mix_0000069_0.00_2_-5_D,0.0,0.413462,0.490385,-0.076923,0.076923
2,mix_0000057_0.00_2_-5_D,0.0,0.355030,0.426036,-0.071006,0.071006
3,mix_0000019_0.00_2_0_P,0.0,0.819149,0.888298,-0.069149,0.069149
4,mix_0000055_0.00_2_0_P,0.0,0.597633,0.656805,-0.059172,0.059172
5,mix_0000067_0.00_2_0_P,0.0,0.668269,0.716346,-0.048077,0.048077
6,mix_0000091_0.00_2_0_P,0.0,0.441341,0.402235,0.039106,0.039106
7,mix_0000043_0.00_2_0_P,0.0,0.896739,0.934783,-0.038043,0.038043
8,mix_0000003_0.00_2_7.4_D,0.0,0.209459,0.243243,-0.033784,0.033784
9,mix_0000094_0.00_2_-5_P,0.0,0.960894,0.988827,-0.027933,0.027933



Top 10 largest |WER - MRS_WER| clips for overlap:


,clip_id,overlap,wer,mrs_wer,wer_minus_mrs_wer,abs_diff
0,mix_0000645_0.14_2_-5_D,0.14,0.352941,0.698039,-0.345098,0.345098
1,mix_0003257_0.40_2_7.4_T,0.40,0.440476,0.761905,-0.321429,0.321429
2,mix_0000309_0.14_2_-5_D,0.14,0.566265,0.873494,-0.307229,0.307229
3,mix_0000223_0.14_2_0_P,0.14,0.590164,0.893443,-0.303279,0.303279
4,mix_0003260_0.40_2_0_T,0.40,0.456349,0.753968,-0.297619,0.297619
5,mix_0003255_0.40_2_7.4_D,0.40,0.464286,0.757937,-0.293651,0.293651
6,mix_0003252_0.40_2_None_D,0.40,0.472222,0.753968,-0.281746,0.281746
7,mix_0003253_0.40_2_None_P,0.40,0.472222,0.753968,-0.281746,0.281746
8,mix_0003254_0.40_2_None_T,0.40,0.472222,0.753968,-0.281746,0.281746
9,mix_0001398_0.20_2_0_D,0.20,0.601810,0.882353,-0.280543,0.280543


In [40]:
# Count rows where standard WER is higher than MRS-WER, and vice versa.
if 'df_comparison' in globals():
    cols = set(df_comparison.columns)
    if {'wer', 'mrs_wer'}.issubset(cols):
        a, b = 'wer', 'mrs_wer'
    elif {'WER', 'MRS_WER'}.issubset(cols):
        a, b = 'WER', 'MRS_WER'
    else:
        a = b = None

    if a is not None:
        temp = df_comparison[[a, b]].dropna()
        wer_higher = int((temp[a] > temp[b]).sum())
        mrs_higher = int((temp[b] > temp[a]).sum())
        equal_count = int((temp[a] == temp[b]).sum())

        print(f"{a} > {b}: {wer_higher}")
        print(f"{b} > {a}: {mrs_higher}")
        print(f"Equal: {equal_count}")
    else:
        print("df_comparison exists, but expected columns were not found.")
        print("Columns:", list(df_comparison.columns))
else:
    print("df_comparison was not found.")

WER > MRS_WER: 9
MRS_WER > WER: 85
Equal: 0


In [2]:
from pathlib import Path
import json
import pandas as pd

# Load only wav2vec2 results; pick a file that contains both WER and MRS_WER.
candidate_files = [
    Path('WER_Results_by_Model_1/wav2vec2_wer_results.json'),
    Path('../WER_Results_by_Model_1/wav2vec2_wer_results.json'),
    Path('/home/jamin/Year3Proj/WER_Results_by_Model_1/wav2vec2_wer_results.json'),
    Path('WER_Results_by_Model/wav2vec2_wer_results.json'),
    Path('../WER_Results_by_Model/wav2vec2_wer_results.json'),
    Path('/home/jamin/Year3Proj/WER_Results_by_Model/wav2vec2_wer_results.json'),
]

existing_files = [p for p in candidate_files if p.exists() and p.is_file()]
if not existing_files:
    raise FileNotFoundError('Could not find wav2vec2_wer_results.json in expected locations.')

rows = []
result_file = None
model_name = 'wav2vec2'

for file_path in existing_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)

    model_name = payload.get('model', 'wav2vec2')
    candidate_rows = []
    for item in payload.get('results', []):
        if item.get('status') != 'success':
            continue

        metrics = item.get('metrics', {})
        wer_val = metrics.get('wer')
        mrs_val = metrics.get('mrs_wer')
        if wer_val is None or mrs_val is None:
            continue

        candidate_rows.append({
            'model': model_name,
            'clip_id': item.get('clip_id', ''),
            'WER': float(wer_val),
            'MRS_WER': float(mrs_val),
        })

    if candidate_rows:
        rows = candidate_rows
        result_file = file_path
        break

if not rows:
    checked = '\n'.join(str(p.resolve()) for p in existing_files)
    raise ValueError('Found wav2vec2 file(s), but none contained both WER and MRS_WER metrics. Checked:\n' + checked)

df_comparison = pd.DataFrame(rows)
wer_higher = int((df_comparison['WER'] > df_comparison['MRS_WER']).sum())
mrs_higher = int((df_comparison['MRS_WER'] > df_comparison['WER']).sum())
equal_count = int((df_comparison['WER'] == df_comparison['MRS_WER']).sum())

print(f'Loaded from: {result_file.resolve()}')
print(f'Model: {model_name}')
print(f'Rows loaded: {len(df_comparison)}')
print(f'WER > MRS_WER: {wer_higher}')
print(f'MRS_WER > WER: {mrs_higher}')
print(f'WER == MRS_WER: {equal_count}')

Loaded from: /home/jamin/Year3Proj/WER_Results_by_Model/wav2vec2_wer_results.json
Model: wav2vec2
Rows loaded: 3720
WER > MRS_WER: 3006
MRS_WER > WER: 253
WER == MRS_WER: 461


In [3]:
from pathlib import Path
import json
import pandas as pd

# Top 10 clips where MRS_WER is lower than WER, with clip id and both transcripts.
if 'df_comparison' not in globals():
    raise NameError('df_comparison is not loaded. Run the earlier comparison cell first.')

required_cols = {'clip_id', 'WER', 'MRS_WER'}
missing = required_cols - set(df_comparison.columns)
if missing:
    raise ValueError(f'df_comparison is missing columns: {sorted(missing)}')

# Load reference transcripts from the manifest.
manifest_path_candidates = [
    Path('Output/manifest.csv'),
    Path('../Output/manifest.csv'),
    Path('/home/jamin/Year3Proj/Output/manifest.csv'),
]
manifest_path = next((p for p in manifest_path_candidates if p.exists()), None)
if manifest_path is None:
    raise FileNotFoundError('Could not find Output/manifest.csv')

manifest_df = pd.read_csv(manifest_path)
if 'clip_id' not in manifest_df.columns or 'transcript' not in manifest_df.columns:
    raise ValueError('manifest.csv must contain clip_id and transcript columns')

def flatten_reference_transcript(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                parts.append(str(item[1]).strip())
            elif isinstance(item, str):
                parts.append(item.strip())
        return ' '.join(part for part in parts if part).strip()
    return ''

manifest_lookup = {}
for _, row in manifest_df.iterrows():
    manifest_lookup[row['clip_id']] = flatten_reference_transcript(row['transcript'])

# Load wav2vec2 hypotheses from ASR_transcriptions.json and flatten them into text.
asr_path_candidates = [
    Path('ASR_transcriptions.json'),
    Path('../ASR_transcriptions.json'),
    Path('/home/jamin/Year3Proj/ASR_transcriptions.json'),
]
asr_path = next((p for p in asr_path_candidates if p.exists()), None)
if asr_path is None:
    raise FileNotFoundError('Could not find ASR_transcriptions.json')

with open(asr_path, 'r', encoding='utf-8') as f:
    asr_payload = json.load(f)

def flatten_model_transcript(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                parts.append(str(item[1]).strip())
            elif isinstance(item, str):
                parts.append(item.strip())
        return ' '.join(part for part in parts if part).strip()
    return ''

wav_lookup = {}
for clip, item in asr_payload.items():
    transcript_block = item.get('transcript', {}) if isinstance(item, dict) else {}
    wav_lookup[clip] = flatten_model_transcript(transcript_block.get('wav2vec2'))

# Build the top-10 table.
summary_df = df_comparison.copy()
summary_df['Difference'] = summary_df['WER'] - summary_df['MRS_WER']
summary_df = summary_df[summary_df['Difference'] > 0].sort_values('Difference', ascending=False).head(10).copy()
summary_df['Reference Transcript'] = summary_df['clip_id'].map(manifest_lookup)
summary_df['Wav2Vec2 Transcript'] = summary_df['clip_id'].map(wav_lookup)

# Keep the output compact and easy to scan.
out = summary_df[[
    'clip_id',
    'WER',
    'MRS_WER',
    'Difference',
    'Reference Transcript',
    'Wav2Vec2 Transcript',
]]

print(f'Loaded manifest from: {manifest_path.resolve()}')
print(f'Loaded ASR transcripts from: {asr_path.resolve()}')
print('\nTop 10 clips where MRS_WER < WER:')
display(out.reset_index(drop=True))

Loaded manifest from: /home/jamin/Year3Proj/Output/manifest.csv
Loaded ASR transcripts from: /home/jamin/Year3Proj/ASR_transcriptions.json

Top 10 clips where MRS_WER < WER:


,clip_id,WER,MRS_WER,Difference,Reference Transcript,Wav2Vec2 Transcript
0,mix_0002937_0.40_2_-5_D,0.518692,0.355140,0.163551,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...
1,mix_0002928_0.40_2_None_D,0.504673,0.350467,0.154206,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...
2,mix_0002929_0.40_2_None_P,0.504673,0.350467,0.154206,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...
3,mix_0002930_0.40_2_None_T,0.504673,0.350467,0.154206,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...
4,mix_0002720_0.40_2_0_T,0.452991,0.303419,0.149573,"[('135842', 'AND FORGOTTEN TO BRUSH HIS HAIR I...",AND FORGOTTEN TO BRUSH HIS HAIR IT POINTED EVE...
5,mix_0002717_0.40_2_7.4_T,0.470085,0.320513,0.149573,"[('135842', 'AND FORGOTTEN TO BRUSH HIS HAIR I...",AND FORGOTTEN TO BRUSH HIS HAIR IT POINTED EVE...
6,mix_0002931_0.40_2_7.4_D,0.528037,0.383178,0.144860,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...
7,mix_0002898_0.40_2_0_D,0.493088,0.350230,0.142857,"[('142776', 'AND THE SAME QUANTITY OF WATER ST...",AND THE SAME QUANTITY OF WATER STEW THEM FOR O...
8,mix_0002895_0.40_2_7.4_D,0.511521,0.368664,0.142857,"[('142776', 'AND THE SAME QUANTITY OF WATER ST...",AND THE SAME QUANTITY OF WATER STEW THEM FOR O...
9,mix_0002933_0.40_2_7.4_T,0.500000,0.359813,0.140187,"[('86815', 'AT ONE OF THE OPEN WINDOWS', 0.0, ...",AT ONE OF THE OPEN WINDOWS WITH A GLOOMY EXPRE...


In [3]:
from faster_whisper import WhisperModel
import os
from pathlib import Path
faster_whiper_model = WhisperModel("large-v3", device="cuda", compute_type="int8")

segments, info = faster_whiper_model.transcribe("/home/jamin/Year3Proj/Output/audio/mix_0000002_0.00_2_None_T.wav")

In [5]:
for segment in segments:
    print(segment)

Segment(id=1, seek=0, start=0.0, end=15.68, text=" he must be very clever and know a great deal for to go up very high which she could never do was the caterpillar's idea of perfect glory now in the neighbouring corn-field there lived a lark", tokens=[50365, 415, 1633, 312, 588, 13494, 293, 458, 257, 869, 2028, 337, 281, 352, 493, 588, 1090, 597, 750, 727, 1128, 360, 390, 264, 44982, 30635, 311, 1558, 295, 2176, 11924, 586, 294, 264, 19755, 278, 9046, 12, 7610, 456, 5152, 257, 287, 809, 51149], avg_logprob=-0.04367519930956212, compression_ratio=1.4074074074074074, no_speech_prob=1.3113021850585938e-05, words=None, temperature=0.0)
Segment(id=2, seek=1568, start=15.68, end=30.4, text=' i slipped up the narrow staircase and had almost touched his finger in the significant action of scent which amabel awaited with breathless expectation when was it miracle or only the suggestion of his better nature', tokens=[50365, 741, 28989, 493, 264, 9432, 35359, 293, 632, 1920, 9828, 702, 5984, 294,

In [6]:
import meeteval

wer = meeteval.wer.wer.orc.orc_word_error_rate(
    reference=['The quick brown fox', 'jumps over the lazy dog'],
    hypothesis=['The kwick brown fox', 'jump over lazy ']
)

print(wer.error_rate)
print(wer.errors, wer.insertions, wer.deletions, wer.substitutions)
print(wer.assignment)

Assuming sort=False because timestamps are missing in reference.
Assuming sort=False because timestamps are missing in hypothesis.


0.4444444444444444
4 0 2 2
(0, 1)


In [13]:
hyp = "A week later Marguerite was settled in her country house, and I was installed at Pointe du Jour. Then began an existence which I shall have some difficulty in describing to you. It was a catastrophe that would have completely vanquished many an inventor, but the Count was saved the gall of defeat. His flight, which was remarkable, inasmuch as he had covered 380 miles... You can report this scene to the Duke, and you can add that we have no longer need of him.\" From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth, Duke was never referred to. From that day forth and equipped with two motors developing 170 horsepower. Upon completion, it was submitted to several preliminary flights, which were so eminently successful that the inventor decided to make a trial trip under conditions closely analogous"

In [11]:
ref = "a week later marguerite was settled in her country house and i was installed at point du jour then began an existence which i shall have some difficulty in describing to you it was reduced to a tangled mass of metallic framework it was a catastrophe that would have completely vanquished many an inventor but the count was saved the gall of defeat his flight which was remarkable inasmuch as he had covered three hundred eighty miles said she turning to prudence and speaking in a broken voice you can report this scene to the duke and you can add that we have no longer need of him from that day forth the duke was never referred to had been eliminated while the ship followed improved lines in its mechanical and structural details the trials with this vessel commenced on november thirtieth nineteen o five but ill luck had not been eluded would you like me to speak to armand marguerite seemed to be thinking for she answered nothing my heart beat violently while i waited for her reply no she answered i will not leave armand and equipped with two motors developing one hundred seventy horse power upon completion it was submitted to several preliminary flights which were so eminently successful that the inventor decided to make a trial trip under conditions closely analogous"

In [14]:
from meeteval.wer.wer.siso import siso_word_error_rate 
siso_word_error_rate(reference=ref, hypothesis=hyp).error_rate

0.9466666666666667

In [16]:
from wer_helper import wer

wer(ref = ref, hyp = hyp)

0.8177777777777778